# FIREWATCH AI — Cross-Satellite Candidate Association

**Stage 05.** Explore possible cross-platform associations for a selectable observation month. This method emits evidence-bearing pairwise candidates only; it does not merge records, claim a shared fire, or produce a probability.

The default 2025-10 period is an analysis slice for threshold exploration, not a tuned or validated operating point. Change `YEAR` and `MONTH` below to inspect another period.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / '2025').is_dir() and (p / '2026').is_dir(): return p
    raise FileNotFoundError('Cannot locate project root')
ROOT=find_root(); sys.path.insert(0,str(ROOT))
from src.firewatch.association import find_candidate_links
DATA=ROOT/'data'/'processed'/'observations.parquet'
YEAR=2025; MONTH=10
MAX_DISTANCE_KM=5.0; MAX_TIME_HOURS=12.0
print('Study period:',f'{YEAR}-{MONTH:02d}','; source:',DATA)

## Load the selected period

Only normalized rows from the requested calendar month are loaded for this exploratory pass. The raw source files and full normalized dataset remain unchanged.

In [ ]:
columns=['observation_id','satellite','product_code','latitude_num','longitude_num','observation_datetime_utc','validation_status']
observations=pd.read_parquet(DATA,columns=columns)
month_start=pd.Timestamp(year=YEAR,month=MONTH,day=1,tz='UTC')
month_end=month_start+pd.offsets.MonthBegin(1)
period=observations.loc[observations.observation_datetime_utc.ge(month_start)&observations.observation_datetime_utc.lt(month_end)].copy()
display(period.groupby(['product_code','satellite','validation_status']).size().rename('records').reset_index())
print(f'Rows in selected month: {len(period):,}')

## Generate a broad candidate pool

The broad window is a search envelope for sensitivity analysis. Every link stores the measured great-circle distance and absolute UTC time difference. Platform pairs are considered across adjacent UTC dates so that detections near midnight are not excluded by date partitioning.

In [ ]:
links=find_candidate_links(period,max_distance_km=MAX_DISTANCE_KM,max_time_delta_hours=MAX_TIME_HOURS)
print(f'Candidate links in broad envelope: {len(links):,}')
display(links.groupby(['link_family','left_satellite','right_satellite']).size().rename('candidate_links').reset_index() if not links.empty else 'No candidate links for this month/window.')
if not links.empty:
    output=ROOT/'data'/'processed'/f'candidate_links_{YEAR}_{MONTH:02d}.parquet'
    output.parent.mkdir(parents=True,exist_ok=True)
    links.to_parquet(output,index=False)
    print('Evidence table saved:',output)

## Threshold sensitivity

Counts below are candidate observation pairs, not unique fires. Tightening thresholds reduces candidates, but does not by itself prove higher accuracy. Reference labels or expert-reviewed samples are needed to estimate false matches and missed associations.

In [ ]:
scenarios=[(0.5,1),(1,3),(2,6),(5,12)]
sensitivity=[]
for distance_km,time_hours in scenarios:
    selected=links.loc[links.distance_km.le(distance_km)&links.time_difference_minutes.le(time_hours*60)]
    detections=pd.unique(pd.concat([selected.left_observation_id,selected.right_observation_id],ignore_index=True)) if not selected.empty else []
    sensitivity.append({'max_distance_km':distance_km,'max_time_hours':time_hours,
                        'candidate_pairs':len(selected),'observations_in_candidate_pairs':len(detections)})
sensitivity_df=pd.DataFrame(sensitivity)
display(sensitivity_df)
ax=sensitivity_df.plot(x='max_distance_km',y='candidate_pairs',marker='o',figsize=(8,4),legend=False)
ax.set_title('Candidate-link count under alternative thresholds')
ax.set_xlabel('Maximum separation (km)'); ax.set_ylabel('Candidate observation pairs')
ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## Inspect candidate evidence

Review individual associations and a small spatial sample. A line indicates only that two detections meet the selected candidate rules. It is not a fire perimeter or a confirmed event boundary.

In [ ]:
if not links.empty:
    display(links.sort_values(['distance_km','time_difference_minutes']).head(25))
    coords=period.set_index('observation_id')[['longitude_num','latitude_num']]
    sample=links.nsmallest(min(150,len(links)),'distance_km')
    fig,ax=plt.subplots(figsize=(8,7))
    for row in sample.itertuples():
        a=coords.loc[row.left_observation_id]; b=coords.loc[row.right_observation_id]
        ax.plot([a.longitude_num,b.longitude_num],[a.latitude_num,b.latitude_num],color='#e87524',alpha=.25,linewidth=.6)
    ax.set_title('Nearest candidate links (sample)')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()
else:
    print('No links to inspect. Try a different month or broaden the search window for exploratory analysis.')

## Interpretation and next decision

The next implementation step is to evaluate candidate-link quality and determine whether a conservative one-to-one association layer is useful. Do not turn this graph into connected event clusters without guarding against transitive chaining (A near B, B near C, but A not near C). Preserve all raw detections and publish the thresholds and evidence whenever a calendar is derived.